<a href="https://colab.research.google.com/github/Brandeis-Visual-Analytics/COSI-165b-pas/blob/main/pa1/pa1_linear_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COSI 165b Deep Learning

Programming Assignment 1: Linear Models

Brandeis University

August 31, 2026

**Check Moodle for Due Date**.  All assignments are due at 11:59 PM on the date listed on Moodle.  Check the syllabus for late submission policy.

**Any AI usage must be reported** at the [unique disclosure form for this assignment](https://genai.cs.brandeis.edu/prompt/1e1560e7-8a45-450f-b8a4-31eb19cf1b08).  AI usage will not affect your grade; we are gathering this information to help us design future instances of the course.

## From pixels to learned features

**Name:** _your name_

This is an **individual** assignment. Make sure that you **File -> Save a copy in Drive** first (so your work persists).  Expected behavior for this notebook is that it runs **top to bottom** to produce all answers.  It may be helpful to read along **UDL Chapter 3** (shallow networks) as you complete this assignment.

**What you'll do:** train a linear model on MNIST (it works well), try the *same* model on MNIST-1D (it fails), add a hidden layer (it recovers), and then look inside that hidden layer to see the features it learned.

**Submitting:** in Colab, do **File -> Save a copy in Drive** first (so your work persists). When done, **Share -> General access: Brandeis University, Viewer -> Copy link**, and paste the link into the PA1 assignment on the Moodle assignment. Make sure you've run all cells so your outputs are visible.


## Part 0 - Setup
Run these cells. We provide you a `train()` method implementing the training loop described in lecture.  Note that it uses pytorch's autograd - the library calculates the gradient itself, and after each batch of training data, we move the parameters a little in the opposite direction of the gradient.  Here the batch is the whole training set - we will discuss other options in a future lecture.

In [ ]:
import time, torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0)                      # Needed to make the code reproducible


In [ ]:
def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

def train(model, X, y, Xt, yt, lr=0.5, epochs=200, log_every=50):
    """Manual full-batch gradient descent. Logs train/test loss and accuracy."""
    loss_fn = nn.CrossEntropyLoss()
    hist = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        logits = model(X)
        loss = loss_fn(logits, y)
        model.zero_grad()
        loss.backward()
        with torch.no_grad():                     # ---- manual gradient-descent step ----
            for p in model.parameters():
                p -= lr * p.grad
        with torch.no_grad():
            hist['train_loss'].append(loss.item())
            hist['test_loss'].append(loss_fn(model(Xt), yt).item())
            hist['train_acc'].append((model(X).argmax(1) == y).float().mean().item())
            hist['test_acc'].append((model(Xt).argmax(1) == yt).float().mean().item())
        if log_every and epoch % log_every == 0:
            print(f'epoch {epoch:4d}  train {hist["train_loss"][-1]:.3f}  test {hist["test_loss"][-1]:.3f}  test_acc {hist["test_acc"][-1]:.3f}')
    hist['time'] = time.time() - t0
    return hist

def plot_losses(hist, title=''):
    plt.figure(figsize=(6, 4))
    plt.plot(hist['train_loss'], label='train loss')
    plt.plot(hist['test_loss'],  label='test loss')
    plt.xlabel('epoch'); plt.ylabel('cross-entropy loss'); plt.title(title); plt.legend(); plt.show()


**Load MNIST** (downloads on first run):

In [ ]:
from torchvision import datasets
mnist_tr = datasets.MNIST('.', train=True,  download=True)
mnist_te = datasets.MNIST('.', train=False, download=True)
Xm  = (mnist_tr.data.float() / 255.).reshape(-1, 28*28)   # [60000, 784]
ym  =  mnist_tr.targets
Xmt = (mnist_te.data.float() / 255.).reshape(-1, 28*28)   # [10000, 784]
ymt =  mnist_te.targets
print('MNIST:', Xm.shape, Xmt.shape)


## Part 1 - Logistic regression on MNIST
A **multinomial logistic regression** is just a single linear layer `784 -> 10` (the softmax lives inside `CrossEntropyLoss`). Build it, train it, and report **test accuracy** (expect it to be high).

**A quick note on PyTorch.** PyTorch is the most popular library for building and training deep learning models. You *can* drop down to very low-level code, but for standard pieces `torch.nn` lets you build simple models — like logistic regression — in just a few lines. A logistic-regression model is a single **`nn.Linear`** layer (the softmax is handled inside `CrossEntropyLoss`, so you don't add it yourself).

New to PyTorch? Skim the **[Build the Neural Network](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)** tutorial — it builds a small network on (Fashion-)MNIST and shows exactly how [`nn.Linear`](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html#nn-linear) is used. The [`torch.nn.Linear` reference](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) lists the constructor arguments (`in_features`, `out_features`).

In [ ]:
# TODO: logistic regression is ONE linear layer from 784 inputs to 10 classes.
lr_mnist = None   # <-- replace None with nn.Linear(..., ...)

hist = train(lr_mnist, Xm, ym, Xmt, ymt, lr=0.3, epochs=200)
print('MNIST logistic-regression test accuracy:', accuracy(lr_mnist, Xmt, ymt))
plot_losses(hist_k, 'logistic - MNIST')



**Look at what it learned.** Each of the 10 output classes has a 784-length weight vector. The code below reshape each back to `28x28` and display it - these are the learned 'templates'.  If a weight for a particular pixel is high, that means that it adds to the score of that class, and vice versa.

The color scale is **symmetric around 0 (white)**: **red = a positive weight** (that pixel being bright *raises* the class's score), **blue = negative** (lowers it).

In [ ]:
W = lr_mnist.weight.detach()                    # [10, 784]
vlim = W.abs().max().item()                     # symmetric colour scale around 0
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for k, ax in enumerate(axes.flat):
    im = ax.imshow(W[k].reshape(28, 28), cmap='RdBu_r', vmin=-vlim, vmax=vlim)
    ax.set_title(f'class {k}'); ax.axis('off')
fig.colorbar(im, ax=axes, shrink=0.85, label='weight value')
plt.suptitle('Learned weight "templates"  (red = pixel argues FOR the class, blue = against)')
plt.show()

### Q1. Why is the accuracy so high?
Look at the weight templates above. In 3-5 sentences, explain why a *linear* model does so well on MNIST.

> _Your answer here._

## Part 2 - The same model on KMNIST
[**KMNIST** (Kuzushiji-MNIST)](https://github.com/rois-codh/kmnist) is a drop-in replacement for MNIST - 28x28 grayscale, 10 classes - but the images are **cursive Japanese characters** with far more variation than printed digits. Same shape, harder problem.

Load it, then train the **same logistic-regression model** (`Linear(784, 10)`) and report test accuracy. Expect it to be **noticeably lower than on MNIST**.

In [ ]:
km_tr = datasets.KMNIST('.', train=True,  download=True)
km_te = datasets.KMNIST('.', train=False, download=True)
Xk  = (km_tr.data.float() / 255.).reshape(-1, 28*28)   # [60000, 784]
yk  =  km_tr.targets
Xkt = (km_te.data.float() / 255.).reshape(-1, 28*28)   # [10000, 784]
ykt =  km_te.targets
print('KMNIST:', Xk.shape, Xkt.shape)


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for c, ax in enumerate(axes.flat):
    idx = (yk == c).nonzero()[0].item()
    ax.imshow(Xk[idx].reshape(28, 28), cmap='gray'); ax.set_title(f'class {c}'); ax.axis('off')
fig.suptitle('KMNIST: one example per class (cursive Kuzushiji characters)')
plt.show()


In [ ]:
# TODO: the SAME logistic regression as Part 1 - a single Linear(784, 10) - now on KMNIST.
lr_kmnist = None   # <-- replace with nn.Linear(...)

hist_k = train(lr_kmnist, Xk, yk, Xkt, ykt, lr=0.5, epochs=200)
print('KMNIST logistic-regression test accuracy:', accuracy(lr_kmnist, Xkt, ykt))
plot_losses(hist_k, 'logistic - KMNIST')


### Q2. Why does the same model do worse on KMNIST than on MNIST?
2-4 sentences. (Hint: how much can a single per-class *template* capture when the characters vary a lot?)

> _Your answer here._

## Part 3 - Add a hidden layer
Build a small MLP on MNIST-1D: `Linear(784 -> H) -> ReLU -> Linear(H -> 10)` with **H = 10**. Train and report test accuracy - you should see a clear jump over the linear model.

Now we'll try to find a **better representation** for the data by adding a **hidden layer** of nodes. Note that each layer should be composed of an `nn.Linear()` followed by an **activation function**, `nn.ReLU()`. *(If this is confusing, please look in Chapter 3 of UDL.*

For the code pattern, see the [`nn.ReLU`](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html#nn-relu) and [`nn.Sequential`](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html#nn-sequential) sections of that same tutorial — stacking `Linear -> ReLU -> Linear` inside an `nn.Sequential` is exactly what you need here.

In [ ]:
H = 10
# TODO: build Linear(784->H) -> ReLU -> Linear(H->10)  (hint: nn.Sequential)
mlp10 = None   # <-- replace with nn.Sequential(...)

hist10 = train(mlp10, Xk, yk, Xkt, ykt, lr=0.5, epochs=300)
print('KMNIST MLP (H=10) test accuracy:', accuracy(mlp10, Xkt, ykt))
plot_losses(hist10, 'MLP H=10 - KMNIST (manual GD)')


### Q3. Did the hidden layer help, and what does that suggest it is doing?
2-4 sentences.

> _Your answer here._

## Part 4 - Look inside the hidden layer
Each hidden unit has a 784-length weight vector - reshape it to **28x28** and it's a *filter*: the image pattern that excites that unit. With H=10 we can look at all ten.

In [ ]:
W1 = mlp10[0].weight.detach()          # [10, 784] - one row per hidden unit
vlim = W1.abs().max().item()
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for j, ax in enumerate(axes.flat):
    im = ax.imshow(W1[j].reshape(28, 28), cmap='RdBu_r', vmin=-vlim, vmax=vlim)
    ax.set_title(f'unit {j}'); ax.axis('off')
fig.colorbar(im, ax=axes, shrink=0.85, label='weight value')
plt.suptitle('Hidden-unit filters (H=10)')
plt.show()


### Q4. What do the hidden units respond to?
Pick one or two units and describe, in a couple of sentences, what part/shape of the signal seems to excite them.

*If you can't make as much sense for this one, do your best - this is more representative of datasets you'll be working with.  Answer why you think the filters are less obvious than with MNIST.*

> _Your answer here._

## Part 5 - Make the hidden layer bigger
Now build the same MLP with **H = 100**, train it, and compare against your H = 10 model on **two axes: test accuracy and training time.**

In [ ]:
mlp100 = None
    

hist100 = train(mlp100, Xk, yk, Xkt, ykt, lr=0.5, epochs=300)
print(f'H=10   test acc {accuracy(mlp10,  Xkt, ykt):.3f}   train time {hist10["time"]:.1f}s')
print(f'H=100  test acc {accuracy(mlp100, Xkt, ykt):.3f}   train time {hist100["time"]:.1f}s')


### Q5. Accuracy vs. cost.
How much did going from 10 to 100 hidden units change accuracy? How much did it change training time? Was it worth it? (2-4 sentences.)

> _Your answer here._

### Q6. (Graduate Students Only): Data Shape
The KMNIST dataset was 768 dimensional.  By using a hidden layer of 10 nodes, we forced it to be 10 dimensional.  Then, by using a hidden layer of 100 nodes, we allowed it to embed itself into 100 dimensions before classifying.

Try to find some plot that explains what the true dimensionality of the data is.  You could try using a projection of the KMNIST data.  You could try to look at correlations between the weights.  Why do you think performance increases when we provide more nodes in the hidden layer?  Without having much experience in the semantic meaning of the data, is there any indication of the "shape" of the data?

> _Your answer here._

In [ ]:
# Q6 code here.